[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/deployment.ipynb) [![在 LangChain Academy 中打开](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239303-lesson-8-deployment)

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Deployment

**一句话定位**:把本地跑通的 agent 部署到云端，让它能通过 API 被外部调用。

**回顾**:到目前为止搭的 agent 已经有：
* **act**——让模型调用具体 tool
* **observe**——把 tool 输出传回模型
* **reason**——模型基于 tool 输出推理下一步
* **persist state**——用 in-memory checkpointer 支持可中断的长对话

**本节目标**：把这个 agent 部署到本地 Studio，以及部署到云端。

</div>

## 核心概念

`LangGraph` ——
- Python / JavaScript 库
- 用来搭建 agent workflow

`LangGraph API` ——
- 把 graph 代码打包成服务
- 提供任务队列管理异步操作
- 提供持久化,维护跨多次交互的 state

`LangSmith Deployment`（曾叫 `LangGraph Cloud`）——
- LangGraph API 的托管服务
- 可以直接从 GitHub 仓库部署 graph
- 提供部署后 graph 的监控和链路追踪
- 每个部署有一个唯一访问 URL

`LangSmith Studio`（曾叫 `LangGraph Studio`）——
- LangGraph 应用的 IDE
- 以 API 为后端，支持实时测试、探索 graph
- 可以本地跑,也可以配合云端部署使用(见下文)

`LangGraph SDK` ——
- 用来以编程方式和 LangGraph graph 交互的 Python 库
- 无论 graph 跑在本地还是云端，都提供一致的调用接口
- 支持创建 client、访问 assistant、管理 thread、执行 run

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**:自 2025 年 10 月起,"LangGraph Platform/Cloud" 的官方名字正式统一为 **LangSmith Deployment**,Studio 也统一叫 **LangSmith Studio**。笔记原文里的旧名(`LangGraph Cloud`/`LangGraph Studio`)已经按现在的叫法标注。现在 Studio 不再是一个独立桌面 App,而是直接在 LangSmith UI 里,进入某个 Deployment 后点 Studio 就能打开——跟本地 `langgraph dev` 启动的本地开发服务器配合用，流程本身(本地起服务、浏览器连 Studio UI)基本没变。参考:[LangGraph Platform 介绍](https://docs.langchain.com/langsmith/deployment-quickstart)。

</div>

## Testing Locally(本地测试)

In [1]:
from langgraph_sdk import get_client

In [2]:
# This is the URL of the local development server
URL = "http://127.0.0.1:9999"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

### 看一下当前有哪些可用的 graph

In [4]:
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-05-02T20:13:44.375063+00:00',
  'updated_at': '2026-05-02T20:13:44.375063+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '228f9934-0cdd-5383-92c8-ee8422522cc2',
  'graph_id': 'router',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'router',
  'created_at': '2026-05-02T20:13:44.368624+00:00',
  'updated_at': '2026-05-02T20:13:44.368624+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '28d99cab-ad6c-5342-aee5-400bd8dc9b8b',
  'graph_id': 'simple_graph',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'simple_graph',
  'created_at': '2026-05-02T19:23:23.802717+00:00',
  'updated_at': '2026-05-02T19:23:23.802717+00:00',
  'version': 1,
  'description': None}]

In [3]:
assistants[-3]

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'config': {},
 'context': {},
 'metadata': {'created_by': 'system'},
 'name': 'agent',
 'created_at': '2026-05-02T20:13:44.375063+00:00',
 'updated_at': '2026-05-02T20:13:44.375063+00:00',
 'version': 1,
 'description': None}

In [5]:
# We create a thread for tracking the state of our run
thread = await client.threads.create()

用 [`client.runs.stream`](https://docs.langchain.com/oss/python/langgraph/graph-api/#stream-and-astream) 跑 agent，需要传：

* `thread_id`
* `graph_id`
* `input`
* `stream_mode`

streaming 的细节会在后面的 module 详细讲。这里只需要知道：`stream_mode="values"` 表示每一步执行完后都把 state 的**完整值**流式返回，state 装在 `chunk.data` 里。

In [6]:
from langchain_core.messages import HumanMessage

# Input
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# Stream
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '6eb4d34c-29d9-4e31-9a80-ff048dee6d22'}
{'content': '', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 17, 'prompt_tokens': 135, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_303cbe1edb', 'id': 'chatcmpl-DbBDOAFkeLKs1xmBdEhug7CVrEljJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 'type': 'ai', 'name': None, 'id': 'lc_run--019dea5f-7150-7820-bd34-f52d11cb9c89-0', 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_RMtQkpArYTiWlUrAPmQKzRRM', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metad

## Testing with Cloud(云端测试)

可以通过 LangSmith 部署到云端，具体步骤参考[官方文档](https://docs.langchain.com/langsmith/deployment-quickstart#deploy-from-github-with-langgraph-cloud)。

### 在 GitHub 上新建一个仓库

* 打开你的 GitHub 账号
* 右上角点 "+" → `"New repository"`
* 给仓库起个名字(比如 `langchain-academy`)

### 把你的 GitHub 仓库加成 remote

* 回到课程开始时 clone `langchain-academy` 的那个终端
* 把刚建的 GitHub 仓库加成 remote

```
git remote add origin https://github.com/your-username/your-repo-name.git
```
* push 上去
```
git push -u origin main
```

### 把 LangSmith 连到你的 GitHub 仓库

* 打开 [LangSmith](https://smith.langchain.com/)
* 点左侧面板的 `deployments` 标签
* 点 `+ New Deployment`
* 选刚创建的那个 GitHub 仓库(比如 `langchain-academy`)
* 把 `LangGraph API config file` 指向某个 `studio` 目录下的配置文件
* 比如 module-1 对应：`module-1/studio/langgraph.json`
* 设置好 API key(可以直接从 `module-1/studio/.env` 复制)

![部署设置截图](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fd61c93d48e5d0f47_deployment2.png)

### 跟部署好的 graph 交互

部署好之后有几种交互方式：

* 像之前一样用 SDK
* 用 [LangGraph Studio](https://docs.langchain.com/langsmith/deployment-quickstart#3-test-your-application-in-studio)

![Studio 交互截图](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fa159a09a51d601de_deployment3.png)

在 notebook 里用 SDK 的话，确保设置了 `LANGSMITH_API_KEY` 就行。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**:部署流程就是"推代码到 GitHub → 在 LangSmith(现在统一叫 LangSmith Deployment)里连接仓库、指向 `langgraph.json` → 拿到部署 URL 后用同一套 LangGraph SDK 调用"——本地调试和云端调用用的是同一套 client 接口，几乎不用改代码。

</div>

In [1]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("LANGSMITH_API_KEY")

In [ ]:
# Replace this with the URL of your deployed graph
URL = "https://langchain-academy-8011c561878d50b1883f7ed11b32d720.default.us.langgraph.app"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

In [37]:
# Select the agent
agent = assistants[0]

In [38]:
agent

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'created_at': '2024-08-23T17:58:02.722920+00:00',
 'updated_at': '2024-08-23T17:58:02.722920+00:00',
 'config': {},
 'metadata': {'created_by': 'system'}}

In [40]:
from langchain_core.messages import HumanMessage

# We create a thread for tracking the state of our run
thread = await client.threads.create()

# Input
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# Stream
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {'example': False, 'additional_kwargs': {}, 'response_metadata': {}}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '8ea04559-f7d4-4c82-89d9-c60fb0502f21', 'example': False}
{'content': '', 'additional_kwargs': {'tool_calls': [{'index': 0, 'id': 'call_EQoolxFaaSVU8HrTnCmffLk7', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'multiply'}, 'type': 'function'}]}, 'response_metadata': {'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_3aa7262c27'}, 'type': 'ai', 'name': None, 'id': 'run-b0ea5ddd-e9ba-4242-bb8c-80eb52466c76', 'example': False, 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_EQoolxFaaSVU8HrTnCmffLk7', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': None}
{'content': '6', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'multiply', 'id': '1bf558e7-79ef-4f21-bb66-acafbd04677a', 'tool_call_id': 'c